# Harry Potter RAG — Book 1

A brief example of a simple workflow of a RAG pipeline using the famous *Harry Potter & The Philosopher's Stone* by J.K. Rowling, including:

- Text processing
- Chunking
- Embedding
- VectorDB management
- Q&A
- Evaluation

Disclaimer: The source book is copyrighted and is not included in this repository. This lab is for personal, non-commercial study only — only original code, questions, and metrics are published.

## Import dependencies

In [45]:
import boto3
import pymupdf
import re

## Setup Bedrock for LLM

## Import data (the book)

From `get_toc()` we know the pdf is structured with chapter+1 as the `pymupdf` index number

In [59]:
doc = pymupdf.open("../../../labs/data/harrypotter.pdf")
toc = doc.get_toc()
toc[:25]

[[1, 'Title Page', 2],
 [1, 'Pottermore.com', 4],
 [1, 'Contents', 6],
 [1, 'Harry Potter and the Sorcerer’s Stone', 7],
 [2, 'Chapter 1 - The Boy Who Lived', 12],
 [2, 'Chapter 2 - The Vanishing Glass', 26],
 [2, 'Chapter 3 - The Letters from No One', 37],
 [2, 'Chapter 4 - The Keeper of the Keys', 50],
 [2, 'Chapter 5 - Diagon Alley', 62],
 [2, 'Chapter 6 - The Journey from Platform Nine and Three-quarters', 85],
 [2, 'Chapter 7 - The Sorting Hat', 107],
 [2, 'Chapter 8 - The Potions Master', 123],
 [2, 'Chapter 9 - The Midnight Duel', 133],
 [2, 'Chapter 10 - Halloween', 150],
 [2, 'Chapter 11 - Quidditch', 164],
 [2, 'Chapter 12 - The Mirror of Erised', 176],
 [2, 'Chapter 13 - Nicolas Flamel', 194],
 [2, 'Chapter 14 - Norbert the Norwegian Ridgeback', 205],
 [2, 'Chapter 15 - The Forbidden Forest', 217],
 [2, 'Chapter 16 - Through the Trapdoor', 234],
 [2, 'Chapter 17 - The Man with Two Faces', 257],
 [2, 'Copyright page', 275],
 [1, 'Harry Potter and the Chamber of Secrets', 276]

Pages with "Chapter 2" represents actual book content:

In [25]:
doc[12-1].get_text()[:250]

'M\n \nCHAPTER  ONE\nTHE BOY WHO LIVED\nr. and Mrs. Dursley, of number four, Privet Drive, were proud to say\nthat they were perfectly normal, thank you very much. They were the\nlast people you’d expect to be involved in anything strange or mysterious,\nbec'

## Building index / metadata

In [54]:
# Get book names
book_names = [
    chapter[1] for chapter in toc if chapter[1].startswith("Harry Potter")
]

['Harry Potter and the Sorcerer’s Stone',
 'Harry Potter and the Chamber of Secrets',
 'Harry Potter and the Prisoner of Azkaban',
 'Harry Potter and the Goblet of Fire',
 'Harry Potter and the Order of the Phoenix',
 'Harry Potter and the Half-Blood Prince',
 'Harry Potter and the Deathly Hallows']

In [66]:
_CHAPTER_RE = re.compile(r"^Chapter\s+(\d+)\s*-\s*(.+)$", re.IGNORECASE)
_CONTENT_SPECIAL = {"Prologue", "Epilogue"}
_BACK_MATTER = "Copyright page"

book_names = [t[1] for t in toc if t[1].startswith("Harry Potter")]
index = []
book_num = 0

def _add_section(book_num, chapter_num, name, page):
    index.append({
        "book_num": book_num,
        "book_name": book_names[book_num - 1],
        "chapter_num": chapter_num,      # None for Prologue/Epilogue
        "chapter_name": name,
        "chapter_start_page_num": page,
        "chapter_end_page_num": None,
    })

for level, title, page in toc:
    title = title.strip()

    if title in book_names:              # book title row -> just bump the counter
        book_num += 1
        continue

    if title == _BACK_MATTER:            # back matter CAPS the previous section
        if index and index[-1]["chapter_end_page_num"] is None:
            index[-1]["chapter_end_page_num"] = page - 1
        continue

    if title in _CONTENT_SPECIAL:
        _add_section(book_num, None, title, page)
        continue

    m = _CHAPTER_RE.match(title)
    if m:
        _add_section(book_num, int(m.group(1)), title, page)

# any section not already capped by back matter ends before the next content section
for prev, nxt in zip(index, index[1:]):
    if prev["chapter_end_page_num"] is None:
        prev["chapter_end_page_num"] = nxt["chapter_start_page_num"] - 1

# final section (book 7's Epilogue) if no Copyright follows it
if index and index[-1]["chapter_end_page_num"] is None:
    index[-1]["chapter_end_page_num"] = doc.page_count

index[:5]

[{'book_num': 1,
  'book_name': 'Harry Potter and the Sorcerer’s Stone',
  'chapter_num': 1,
  'chapter_name': 'Chapter 1 - The Boy Who Lived',
  'chapter_start_page_num': 12,
  'chapter_end_page_num': 25},
 {'book_num': 1,
  'book_name': 'Harry Potter and the Sorcerer’s Stone',
  'chapter_num': 2,
  'chapter_name': 'Chapter 2 - The Vanishing Glass',
  'chapter_start_page_num': 26,
  'chapter_end_page_num': 36},
 {'book_num': 1,
  'book_name': 'Harry Potter and the Sorcerer’s Stone',
  'chapter_num': 3,
  'chapter_name': 'Chapter 3 - The Letters from No One',
  'chapter_start_page_num': 37,
  'chapter_end_page_num': 49},
 {'book_num': 1,
  'book_name': 'Harry Potter and the Sorcerer’s Stone',
  'chapter_num': 4,
  'chapter_name': 'Chapter 4 - The Keeper of the Keys',
  'chapter_start_page_num': 50,
  'chapter_end_page_num': 61},
 {'book_num': 1,
  'book_name': 'Harry Potter and the Sorcerer’s Stone',
  'chapter_num': 5,
  'chapter_name': 'Chapter 5 - Diagon Alley',
  'chapter_start_pag

## Chunking text